# Semantic model refresh history loader

Collects the refresh history of every semantic model in the configured workspaces from the Power BI REST API and upserts it into `monitoring.semantic_model_refresh_history` (one row per refresh).

It covers every refresh type: scheduled, on-demand, pipeline (the *Semantic model refresh* activity shows as `ViaEnhancedApi`), API and XMLA. `pipeline_run_history` only sees refreshes a pipeline triggers. For failed enhanced refreshes it also pulls the detail messages, which carry the engine error (e.g. a duplicate value on the "one" side of a relationship).

**Before the first run:** attach the Lakehouse that holds the `monitoring` schema as this notebook's default Lakehouse, and run `semantic_model_refresh_history_ddl.sql` once to create the table and view.

**Schedule:** the API keeps only 20-60 refreshes per model (entries older than 3 days are dropped once there are more than 20), so run at least daily. The `pipeline_run_monitor` pipeline runs it hourly next to the pipeline loader. Every run reads all the history the API still holds, so there is no lookback setting.

In [ ]:
# ---- Config (parameter cell) ----
WORKSPACE_IDS = [
    "f78b1a7f-a951-4a83-a0bb-c071a0451046",   ]

# None = every semantic model in the workspaces above; or a list of model display names
SEMANTIC_MODEL_NAMES = None

# Skip models whose name contains any of these (case-insensitive)
EXCLUDE_NAME_CONTAINS = ["obsolete"]

REFRESH_TABLE = "monitoring.semantic_model_refresh_history"
HEALTH_VIEW   = "monitoring.vw_semantic_model_refresh_health"

In [ ]:
# ---- API helpers ----
import json
import re
import time
from datetime import datetime, timedelta, timezone

import requests
from notebookutils import mssparkutils

FABRIC_API = "https://api.fabric.microsoft.com/v1"
PBI_API    = "https://api.powerbi.com/v1.0/myorg"


def api(method, url, body=None, max_retries=5):
    """Call the Fabric or Power BI REST API; retries on throttling (429) and 5xx."""
    audience = "pbi" if url.startswith(PBI_API) else "https://api.fabric.microsoft.com"
    for attempt in range(max_retries):
        token = mssparkutils.credentials.getToken(audience)
        r = requests.request(method, url, json=body, timeout=60,
                             headers={"Authorization": f"Bearer {token}"})
        if r.status_code == 429 or r.status_code >= 500:
            time.sleep(int(r.headers.get("Retry-After", 5 * 2 ** attempt)))
            continue
        r.raise_for_status()
        return r.json() if r.content else {}
    r.raise_for_status()


def get_paged(url):
    """GET a Fabric list endpoint, following continuationUri."""
    rows = []
    while url:
        page = api("GET", url)
        rows.extend(page.get("value", []))
        url = page.get("continuationUri")
    return rows


def parse_ts(s):
    """API timestamp ('2026-09-10T17:00:03.847Z', with or without Z) -> UTC datetime."""
    if not s:
        return None
    m = re.match(r"(\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2})(?:\.(\d+))?", s)
    frac = (m.group(2) or "0")[:6].ljust(6, "0")
    return datetime.fromisoformat(f"{m.group(1)}.{frac}").replace(tzinfo=timezone.utc)


def service_exception(s):
    """serviceExceptionJson (a JSON string, sometimes plain text) -> dict."""
    if not s:
        return {}
    try:
        d = json.loads(s)
        return d if isinstance(d, dict) else {"errorDescription": s}
    except ValueError:
        return {"errorDescription": s}


def detail_errors(detail):
    """Distinct error messages of an enhanced-refresh detail payload, one per line."""
    msgs = [m.get("message") for m in (detail or {}).get("messages") or []
            if m.get("message") and m.get("type", "Error") == "Error"]
    return "\n".join(dict.fromkeys(msgs)) or None


def refresh_row(ws_id, ws_name, model, r, detail, now):
    start, end = parse_ts(r.get("startTime")), parse_ts(r.get("endTime"))
    exc = service_exception(r.get("serviceExceptionJson"))
    attempts = r.get("refreshAttempts")
    return {
        "semantic_model_id":      model["id"],
        "request_id":             r["requestId"],
        "refresh_id":             r.get("id"),
        "workspace_id":           ws_id,
        "workspace_name":         ws_name,
        "semantic_model_name":    model["displayName"],
        "refresh_type":           r.get("refreshType"),
        "status":                 r.get("status") or "Unknown",
        "extended_status":        r.get("extendedStatus"),
        "start_time_utc":         start,
        "end_time_utc":           end,
        "duration_ms":            int((end - start).total_seconds() * 1000) if start and end else None,
        "attempt_count":          len(attempts) if attempts is not None else None,
        "error_code":             exc.get("errorCode"),
        "error_description":      exc.get("errorDescription"),
        "error_messages":         detail_errors(detail),
        "service_exception_json": r.get("serviceExceptionJson"),
        "attempts_json":          json.dumps(attempts) if attempts is not None else None,
        "detail_json":            json.dumps(detail) if detail is not None else None,
        "raw_json":               json.dumps(r),
        "ingested_at_utc":        now,
        "updated_at_utc":         now,
    }

In [ ]:
# ---- Collect from the API ----
spark.conf.set("spark.sql.session.timeZone", "UTC")
now = datetime.now(timezone.utc)

# Status already stored for refreshes the API can still return (it keeps ~3 days), so the
# detail call runs once per failed refresh rather than on every load.
stored_status = {(r.semantic_model_id, r.request_id): r.status for r in
                 spark.table(REFRESH_TABLE)
                      .where(f"start_time_utc >= TIMESTAMP '{now - timedelta(days=7):%Y-%m-%d %H:%M:%S}'")
                      .select("semantic_model_id", "request_id", "status").collect()}

refresh_rows, errors, warnings = [], [], []

for ws_id in WORKSPACE_IDS:
    ws_name = api("GET", f"{FABRIC_API}/workspaces/{ws_id}")["displayName"]
    models = get_paged(f"{FABRIC_API}/workspaces/{ws_id}/items?type=SemanticModel")
    if SEMANTIC_MODEL_NAMES:
        models = [m for m in models if m["displayName"] in SEMANTIC_MODEL_NAMES]
    skipped = [m["displayName"] for m in models
               if any(x.lower() in m["displayName"].lower() for x in EXCLUDE_NAME_CONTAINS)]
    models = [m for m in models if m["displayName"] not in skipped]
    if skipped:
        print(f"Skipped {len(skipped)} excluded model(s): {', '.join(sorted(skipped))}")

    for model in models:
        base = f"{PBI_API}/groups/{ws_id}/datasets/{model['id']}/refreshes"
        try:
            refreshes = api("GET", base).get("value", [])
        except requests.HTTPError as e:
            errors.append(f"{model['displayName']}: {e}")
            continue

        details = 0
        for r in refreshes:
            detail = None
            newly_failed = (r.get("status") == "Failed"
                            and stored_status.get((model["id"], r["requestId"])) != "Failed")
            # Only enhanced refreshes (pipeline activity / enhanced API) have a detail endpoint
            if newly_failed and r.get("refreshType") == "ViaEnhancedApi":
                try:
                    detail = api("GET", f"{base}/{r['requestId']}")
                    details += 1
                except requests.HTTPError as e:
                    warnings.append(f"{model['displayName']} refresh {r['requestId']} detail: {e}")
            refresh_rows.append(refresh_row(ws_id, ws_name, model, r, detail, now))

        print(f"{model['displayName']:<55} refreshes={len(refreshes):>3}  detail calls={details}")

print(f"\nCollected {len(refresh_rows)} refreshes, {len(errors)} errors, {len(warnings)} warnings")

In [ ]:
# ---- Upsert into the history table ----
def stage(rows, table, keys, view):
    """Temp view shaped exactly like the target table (so MERGE can INSERT *)."""
    schema = spark.table(table).schema
    df = spark.createDataFrame([tuple(r[f.name] for f in schema) for r in rows], schema)
    df.dropDuplicates(keys).createOrReplaceTempView(view)

stage(refresh_rows, REFRESH_TABLE, ["semantic_model_id", "request_id"], "stg_refreshes")

spark.sql(f"""
MERGE INTO {REFRESH_TABLE} AS t
USING stg_refreshes AS s
   ON t.semantic_model_id = s.semantic_model_id
  AND t.request_id        = s.request_id
WHEN MATCHED AND (   t.status <> s.status
                  OR NOT (t.extended_status <=> s.extended_status)
                  OR NOT (t.end_time_utc    <=> s.end_time_utc)
                  OR NOT (t.attempt_count   <=> s.attempt_count)) THEN UPDATE SET
    refresh_id             = s.refresh_id,
    status                 = s.status,
    extended_status        = s.extended_status,
    start_time_utc         = s.start_time_utc,
    end_time_utc           = s.end_time_utc,
    duration_ms            = s.duration_ms,
    attempt_count          = s.attempt_count,
    error_code             = s.error_code,
    error_description      = s.error_description,
    error_messages         = COALESCE(s.error_messages, t.error_messages),
    service_exception_json = s.service_exception_json,
    attempts_json          = s.attempts_json,
    detail_json            = COALESCE(s.detail_json, t.detail_json),
    raw_json               = s.raw_json,
    updated_at_utc         = s.updated_at_utc
WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE done")

In [ ]:
# ---- Last 2 days ----
display(spark.sql(f"""
    SELECT semantic_model_name, start_time_sgt, refresh_type, effective_status,
           duration_minutes, attempt_count, error_detail
    FROM {HEALTH_VIEW}
    WHERE refresh_date_sgt >= date_sub(current_date(), 2)
    ORDER BY start_time_sgt DESC
"""))

In [ ]:
# ---- Last 14 days per model ----
display(spark.sql(f"""
    SELECT semantic_model_name,
           COUNT(*)                                                          AS refreshes,
           COUNT_IF(effective_status = 'Completed')                          AS completed,
           COUNT_IF(effective_status = 'Failed')                             AS failed,
           ROUND(100.0 * COUNT_IF(effective_status = 'Completed')
                 / NULLIF(COUNT_IF(effective_status <> 'Running'), 0), 1)    AS success_pct,
           ROUND(AVG(CASE WHEN effective_status = 'Completed' THEN duration_minutes END), 2) AS avg_completed_minutes,
           MAX(CASE WHEN effective_status = 'Completed' THEN start_time_sgt END)             AS last_success_sgt,
           MAX_BY(error_detail, CASE WHEN effective_status = 'Failed' THEN start_time_sgt END) AS last_error
    FROM {HEALTH_VIEW}
    WHERE refresh_date_sgt >= date_sub(current_date(), 14)
    GROUP BY semantic_model_name
    ORDER BY failed DESC, semantic_model_name
"""))

In [ ]:
# ---- Fail the run if any model's history couldn't be read ----
for w in warnings:
    print(f"WARNING {w}")
if errors:
    # Everything that was collected is already merged; the next run retries the rest.
    raise RuntimeError("Some API calls failed:\n" + "\n".join(errors))
print("Done")